# Companion code for model diagnostics

## Loss curves

In [ ]:
import warnings

import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler

params = [
    {"solver": "sgd", "learning_rate": "constant", "momentum": 0, "learning_rate_init": 0.2},
    {
        "solver": "sgd",
        "learning_rate": "constant",
        "momentum": 0.9,
        "nesterovs_momentum": False,
        "learning_rate_init": 0.2,
    },
    {
        "solver": "sgd",
        "learning_rate": "constant",
        "momentum": 0.9,
        "nesterovs_momentum": True,
        "learning_rate_init": 0.2,
    },
    {"solver": "sgd", "learning_rate": "invscaling", "momentum": 0, "learning_rate_init": 0.2},
    {
        "solver": "sgd",
        "learning_rate": "invscaling",
        "momentum": 0.9,
        "nesterovs_momentum": False,
        "learning_rate_init": 0.2,
    },
    {
        "solver": "sgd",
        "learning_rate": "invscaling",
        "momentum": 0.9,
        "nesterovs_momentum": True,
        "learning_rate_init": 0.2,
    },
    {"solver": "adam", "learning_rate_init": 0.01},
]

labels = [
    "constant learning rate",
    "constant with momentum",
    "constant with Nesterov momentum",
    "inv-scaling learning rate",
    "inv-scaling with momentum",
    "inv-scaling with Nesterov momentum",
    "adam",
]

styles = [
    {"c": "red", "linestyle": "-"},
    {"c": "green", "linestyle": "-"},
    {"c": "blue", "linestyle": "-"},
    {"c": "red", "linestyle": "--"},
    {"c": "green", "linestyle": "--"},
    {"c": "blue", "linestyle": "--"},
    {"c": "black", "linestyle": "-"},
]


def plot_loss_curves(X, y, ax, title):
    X_scaled = MinMaxScaler().fit_transform(X)
    max_iter = 15 if title == "digits" else 400

    for label, param, style in zip(labels, params, styles):
        model = MLPClassifier(random_state=0, max_iter=max_iter, **param)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")
            model.fit(X_scaled, y)
        ax.plot(model.loss_curve_, label=label, **style)

    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")


iris = datasets.load_iris()
X_digits, y_digits = datasets.load_digits(return_X_y=True)
data_sets = [
    (iris.data, iris.target, "iris"),
    (X_digits, y_digits, "digits"),
    (*datasets.make_circles(noise=0.2, factor=0.5, random_state=1), "circles"),
    (*datasets.make_moons(noise=0.3, random_state=0), "moons"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (X_data, y_data, name) in zip(axes.ravel(), data_sets):
    plot_loss_curves(X_data, y_data, ax, name)

fig.legend(axes[0, 0].get_lines(), labels, ncol=3, loc="upper center")
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## Validation curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import ValidationCurveDisplay
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
subset_mask = np.isin(y, [1, 2])
X_binary, y_binary = X[subset_mask], y[subset_mask]

disp = ValidationCurveDisplay.from_estimator(
    SVC(),
    X_binary,
    y_binary,
    param_name="gamma",
    param_range=np.logspace(-6, -1, 5),
    score_type="both",
    n_jobs=2,
    score_name="Accuracy",
)
disp.ax_.set_title("Validation curve for SVM with RBF kernel")
disp.ax_.set_xlabel("gamma")
disp.ax_.set_ylim(0.0, 1.1)
plt.show()

## Learning curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

data = pd.read_excel("../data/real_estate_valuation.xlsx")
data = data.rename(columns={column: column.split()[0] for column in data.columns}).drop(columns="No")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]
train_sizes = np.linspace(0.1, 1.0, 10)


def plot_learning_curve(estimator, title):
    sizes, train_scores, val_scores = learning_curve(
        estimator=estimator,
        X=X,
        y=y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        train_sizes=train_sizes,
    )

    train_rmse = -train_scores.mean(axis=1)
    val_rmse = -val_scores.mean(axis=1)

    plt.figure(figsize=(10, 6))
    plt.plot(sizes, train_rmse, marker="o", label="train")
    plt.plot(sizes, val_rmse, marker="s", label="validation")
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("RMSE")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_learning_curve(DecisionTreeRegressor(random_state=0), "Learning curve: overfitting example")
plot_learning_curve(
    make_pipeline(StandardScaler(), SVR(C=0.25)),
    "Learning curve: underfitting example",
)
plot_learning_curve(
    RandomForestRegressor(max_depth=3, random_state=0),
    "Learning curve: better fit",
)